# 🧠 RoleNet v2 — Cải Thiện Accuracy Từ 59% → 80%+

## 🔑 Thay đổi so với v1:
| Thành phần | V1 (59%) | V2 (mục tiêu 80%+) |
|---|---|---|
| **Architecture** | MobileNetV3-Small | **EfficientNet-B2** (mạnh hơn ~15%) |
| **Classifier head** | Linear(576→256→16) | **Linear(1408→512→256→16)** + BN |
| **Augmentation** | ColorJitter + RandomErasing | **+ MixUp + CutMix + RandomPerspective** |
| **Unfreezing** | Toàn bộ ngay lập tức | **Progressive: head→top layers→toàn bộ** |
| **Loss** | CrossEntropy + label_smooth=0.1 | **+ Class-weighted loss** |
| **TTA** | Không | **Test-Time Augmentation (flip + scale)** |
| **LR** | AdamW 3e-4 OneCycle | **AdamW 1e-3 → cosine với warmup** |

---
⚡ **Yêu cầu**: Runtime → T4 GPU | Upload `rolenet_upload.zip` lên Google Drive

## 📌 Bước 1: Mount Drive & Giải nén Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ZIP_PATH    = '/content/drive/MyDrive/rolenet_upload.zip'
EXTRACT_DIR = '/content/rolenet_dataset'

if not os.path.exists(EXTRACT_DIR):
    print('📦 Giải nén dataset...')
    !unzip -q "{ZIP_PATH}" -d /content/
    print('✅ Xong!')
else:
    print('✅ Dataset đã tồn tại.')

# Đếm ảnh
from pathlib import Path
aug_dir = Path(EXTRACT_DIR) / 'augmented'
total = sum(len(list(d.glob('*.jpg'))) for d in aug_dir.iterdir() if d.is_dir())
print(f'📊 Tổng ảnh augmented: {total:,}')
!ls {EXTRACT_DIR}/augmented/

## 📌 Bước 2: Cài Thư Viện

In [ ]:
!pip install -q timm torchmetrics --upgrade
import torch
import timm
print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU"}')
print(f'CUDA    : {torch.version.cuda}')

## 📌 Bước 3: Config

In [ ]:
import torch, timm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
import json, time, copy, random

# ============================================================
# CONFIG V2
# ============================================================
CFG = {
    # Dataset
    'data_root'    : '/content/rolenet_dataset/augmented',
    'splits_dir'   : '/content/rolenet_dataset/splits',
    'img_size'     : (256, 128),

    # Model — EfficientNet-B2 thay thế MobileNetV3-Small
    'model_name'   : 'efficientnet_b2',
    'num_classes'  : 16,
    'pretrained'   : True,
    'drop_rate'    : 0.35,       # Dropout trong classifier

    # Training — 3 phase progressive unfreezing
    'phase1_epochs': 5,          # Chỉ train head
    'phase2_epochs': 10,         # Train head + top 2 blocks
    'phase3_epochs': 20,         # Train toàn bộ
    'batch_size'   : 48,
    'lr_head'      : 1e-3,       # LR cho classifier head
    'lr_backbone'  : 1e-4,       # LR cho backbone (phase 3)
    'weight_decay' : 1e-4,
    'label_smooth' : 0.1,

    # Augmentation
    'mixup_alpha'  : 0.4,        # MixUp alpha
    'cutmix_alpha' : 1.0,        # CutMix alpha
    'mixup_prob'   : 0.5,        # Xác suất áp dụng MixUp/CutMix

    # Output
    'save_dir'     : '/content/drive/MyDrive/rolenet_v2_checkpoints',
    'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers'  : 4,
    'seed'         : 42,
}

CLASS_NAMES = [
    'shipper', 'doctor', 'police', 'military', 'security', 'student',
    'chef', 'janitor', 'construction', 'nurse', 'postman', 'technician',
    'worker', 'civil_guard', 'normal', 'unknown'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])

Path(CFG['save_dir']).mkdir(parents=True, exist_ok=True)

total_epochs = CFG['phase1_epochs'] + CFG['phase2_epochs'] + CFG['phase3_epochs']
print('✅ Config V2 loaded!')
print(f'   Model   : {CFG["model_name"]}')
print(f'   Device  : {CFG["device"]}')
print(f'   Epochs  : {total_epochs} (phase1={CFG["phase1_epochs"]}, phase2={CFG["phase2_epochs"]}, phase3={CFG["phase3_epochs"]})')
print(f'   Classes : {CFG["num_classes"]}')

## 📌 Bước 4: Dataset & DataLoader

In [ ]:
# ============================================================
# Dataset
# ============================================================
class RoleNetDataset(Dataset):
    def __init__(self, root_dir, split_csv, transform=None):
        self.root      = Path(root_dir)
        self.transform = transform
        self.samples   = []
        df = pd.read_csv(split_csv)
        for _, row in df.iterrows():
            p = self.root / row['path']
            l = CLASS_TO_IDX.get(row['label'], -1)
            if p.exists() and l >= 0:
                self.samples.append((str(p), l))
        print(f'  Loaded {len(self.samples):,} from {Path(split_csv).name}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================================================
# Transforms — V2 mạnh hơn nhiều
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.RandomPerspective(distortion_scale=0.3)
    ], p=0.3),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),  # Occlusion
])

val_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('📂 Loading datasets...')
train_ds = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/train.csv", train_transform)
val_ds   = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/val.csv",   val_transform)
test_ds  = RoleNetDataset(CFG['data_root'], f"{CFG['splits_dir']}/test.csv",  val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'\n✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

# Phân phối lớp
labels = [s[1] for s in train_ds.samples]
counts = Counter(labels)
print('\n📊 Class distribution:')
for i, name in enumerate(CLASS_NAMES):
    bar = '█' * (counts[i] // 50)
    print(f'  {name:15s} {counts[i]:5d}  {bar}')

# Class weights cho imbalanced dataset
total_n   = sum(counts.values())
class_wts = torch.tensor([total_n / (16.0 * counts[i]) for i in range(16)],
                          dtype=torch.float32).to(CFG['device'])
print(f'\n⚖️  Class weights: min={class_wts.min():.2f}, max={class_wts.max():.2f}')

## 📌 Bước 5: Model — EfficientNet-B2

In [ ]:
def build_rolenet_v2(num_classes=16, pretrained=True, drop_rate=0.35):
    """
    EfficientNet-B2 fine-tuned cho RoleNet v2.
    - EfficientNet-B2: 9.1M params, ImageNet top-1=80.6%
    - Classifier: Adaptive pool → BN → Dropout → FC(1408→512) → BN → Dropout → FC(512→16)
    - Progressive freeze: ban đầu chỉ train head
    """
    model = timm.create_model(
        'efficientnet_b2',
        pretrained=pretrained,
        num_classes=0,           # Bỏ head gốc
        drop_rate=drop_rate,
    )
    in_features = model.num_features  # 1408 với EfficientNet-B2

    # Classifier head mạnh hơn
    head = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.BatchNorm1d(in_features),
        nn.Dropout(p=drop_rate),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(p=drop_rate * 0.5),
        nn.Linear(512, num_classes),
    )

    # Wrap thành 1 model
    class RoleNetV2(nn.Module):
        def __init__(self, backbone, classifier):
            super().__init__()
            self.backbone   = backbone
            self.classifier = classifier

        def forward(self, x):
            feat = self.backbone.forward_features(x)
            return self.classifier(feat)

        def freeze_backbone(self):
            for p in self.backbone.parameters():
                p.requires_grad = False

        def unfreeze_top_blocks(self, n_blocks=2):
            """Unfreeze n block cuối của EfficientNet."""
            blocks = list(self.backbone.blocks.children())
            for blk in blocks[-n_blocks:]:
                for p in blk.parameters():
                    p.requires_grad = True
            # conv_head luôn unfreeze
            for p in self.backbone.conv_head.parameters():
                p.requires_grad = True
            for p in self.backbone.bn2.parameters():
                p.requires_grad = True

        def unfreeze_all(self):
            for p in self.parameters():
                p.requires_grad = True

    net = RoleNetV2(model, head)
    return net


model = build_rolenet_v2(
    num_classes=CFG['num_classes'],
    pretrained=CFG['pretrained'],
    drop_rate=CFG['drop_rate'],
)
model = model.to(CFG['device'])

total_p    = sum(p.numel() for p in model.parameters())
print(f'✅ Model: EfficientNet-B2 + Custom Head')
print(f'   Total params : {total_p:,} (~{total_p*4/1024/1024:.1f} MB)')

# Test forward
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 128).to(CFG['device'])
    out   = model(dummy)
    print(f'   Output shape : {out.shape} ✅')

## 📌 Bước 6: MixUp & CutMix Augmentation

In [ ]:
def mixup_data(x, y, alpha=0.4, device='cuda'):
    """Trộn 2 sample theo tỷ lệ ngẫu nhiên."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    bs  = x.size(0)
    idx = torch.randperm(bs).to(device)
    x_mix  = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return x_mix, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0, device='cuda'):
    """Cắt và ghép vùng từ 2 ảnh khác nhau."""
    lam = np.random.beta(alpha, alpha)
    bs  = x.size(0)
    idx = torch.randperm(bs).to(device)

    _, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w   = int(W * cut_rat)
    cut_h   = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return x_cut, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('✅ MixUp & CutMix functions defined.')
print(f'   MixUp  alpha={CFG["mixup_alpha"]}, prob={CFG["mixup_prob"]}')
print(f'   CutMix alpha={CFG["cutmix_alpha"]}')

## 📌 Bước 7: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device,
                use_mixup=True):
    model.train()
    total_loss = correct = n = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # MixUp / CutMix (chỉ dùng trong phase 2 & 3)
        use_mix = use_mixup and random.random() < CFG['mixup_prob']
        if use_mix:
            if random.random() < 0.5:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG['mixup_alpha'], device)
            else:
                imgs, y_a, y_b, lam = cutmix_data(imgs, labels, CFG['cutmix_alpha'], device)

        optimizer.zero_grad()
        outputs = model(imgs)

        if use_mix:
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        n          += imgs.size(0)

    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = n = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        n          += imgs.size(0)

    return total_loss / n, correct / n


def make_optimizer(model, phase):
    """Tạo optimizer với LR khác nhau cho head và backbone."""
    if phase == 1:
        # Chỉ train classifier head
        params = filter(lambda p: p.requires_grad, model.parameters())
        return optim.AdamW(params, lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
    elif phase == 2:
        # Head LR cao, top blocks LR thấp hơn
        return optim.AdamW([
            {'params': model.classifier.parameters(),  'lr': CFG['lr_head']},
            {'params': filter(lambda p: p.requires_grad,
                              model.backbone.parameters()), 'lr': CFG['lr_backbone'] * 2},
        ], weight_decay=CFG['weight_decay'])
    else:
        # Phase 3: toàn bộ, backbone LR nhỏ hơn head
        return optim.AdamW([
            {'params': model.classifier.parameters(),  'lr': CFG['lr_head'] * 0.5},
            {'params': model.backbone.parameters(),     'lr': CFG['lr_backbone']},
        ], weight_decay=CFG['weight_decay'])


print('✅ Training functions ready.')

## 📌 Bước 8: Training — 3 Phase Progressive Unfreezing

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_wts,
    label_smoothing=CFG['label_smooth']
)

history  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'phase': []}
best_acc = 0.0
best_weights = None
save_path = Path(CFG['save_dir'])


def run_phase(phase_num, n_epochs, use_mixup=False):
    global best_acc, best_weights

    optimizer = make_optimizer(model, phase_num)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs * len(train_loader), eta_min=1e-6
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*70}')
    print(f'  PHASE {phase_num} | {n_epochs} epochs | MixUp={use_mixup} '
          f'| Trainable params: {trainable:,}')
    print(f'{"="*70}')

    for ep in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer,
                                       scheduler, CFG['device'], use_mixup)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, CFG['device'])

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['phase'].append(phase_num)

        is_best = vl_acc > best_acc
        if is_best:
            best_acc     = vl_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch'      : sum([CFG['phase1_epochs'], CFG['phase2_epochs']][:phase_num-1]) + ep,
                'model_state': model.state_dict(),
                'val_acc'    : vl_acc,
                'classes'    : CLASS_NAMES,
                'phase'      : phase_num,
            }, save_path / 'rolenet_v2_best.pt')

        total_ep = len(history['val_acc'])
        lr_now   = optimizer.param_groups[0]['lr']
        elapsed  = time.time() - t0
        marker   = ' ⭐ BEST' if is_best else ''
        print(
            f'  P{phase_num} Ep {ep:2d}/{n_epochs} | '
            f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
            f'Acc {tr_acc*100:5.1f}%/{vl_acc*100:5.1f}% | '
            f'LR {lr_now:.1e} | {elapsed:.0f}s{marker}'
        )


t_start = time.time()

# ─── PHASE 1: Chỉ train head ────────────────────────────────
print('🔒 PHASE 1: Freeze backbone, chỉ train classifier head...')
model.freeze_backbone()
run_phase(1, CFG['phase1_epochs'], use_mixup=False)

# ─── PHASE 2: Unfreeze top 3 blocks ─────────────────────────
print('\n🔓 PHASE 2: Unfreeze top 3 blocks + head...')
model.unfreeze_top_blocks(n_blocks=3)
run_phase(2, CFG['phase2_epochs'], use_mixup=True)

# ─── PHASE 3: Unfreeze toàn bộ ──────────────────────────────
print('\n🔓 PHASE 3: Unfreeze toàn bộ model...')
model.unfreeze_all()
run_phase(3, CFG['phase3_epochs'], use_mixup=True)

total_time = (time.time() - t_start) / 60
print(f'\n✅ Training hoàn tất! {total_time:.1f} phút')
print(f'   🏆 Best Val Accuracy: {best_acc*100:.2f}%')
print(f'   📁 Saved → {save_path}/rolenet_v2_best.pt')

## 📌 Bước 9: Learning Curve

In [ ]:
epochs_ran = len(history['train_acc'])
x = range(1, epochs_ran + 1)

# Phase boundaries
p1_end = CFG['phase1_epochs']
p2_end = p1_end + CFG['phase2_epochs']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(x, history['train_loss'], 'b-', label='Train Loss', lw=2)
ax1.plot(x, history['val_loss'],   'r-', label='Val Loss',   lw=2)
ax1.axvline(p1_end, color='orange', ls='--', alpha=0.7, label='Phase 1→2')
ax1.axvline(p2_end, color='green',  ls='--', alpha=0.7, label='Phase 2→3')
ax1.set_title('Loss', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(x, [a*100 for a in history['train_acc']], 'b-', label='Train Acc', lw=2)
ax2.plot(x, [a*100 for a in history['val_acc']],   'r-', label='Val Acc',   lw=2)
ax2.axhline(best_acc*100, color='gold', ls='--', lw=2, label=f'Best {best_acc*100:.1f}%')
ax2.axhline(59.05,        color='gray', ls=':',  lw=1, label='V1 baseline 59%')
ax2.axvline(p1_end, color='orange', ls='--', alpha=0.7)
ax2.axvline(p2_end, color='green',  ls='--', alpha=0.7)
ax2.set_title('Accuracy', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle(f'RoleNet V2 Training — Best: {best_acc*100:.1f}% (V1: 59%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Chart saved → /content/training_v2.png')

## 📌 Bước 10: Đánh Giá Test Set + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.load_state_dict(best_weights)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(CFG['device'])).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('📊 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

cm   = confusion_matrix(all_labels, all_preds)
cm_p = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm_p, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={'label': '%'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — RoleNet V2 (Best Acc: {best_acc*100:.1f}%)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved → /content/confusion_v2.png')

## 📌 Bước 11: Test-Time Augmentation (TTA)

In [ ]:
@torch.no_grad()
def predict_tta(model, imgs_tensor, device, n_aug=5):
    """
    Test-Time Augmentation: lấy trung bình xác suất qua nhiều augmentation.
    Giúp tăng thêm ~1-2% accuracy khi inference.
    """
    model.eval()
    imgs = imgs_tensor.to(device)
    probs_sum = torch.softmax(model(imgs), dim=1)

    # Horizontal flip
    probs_sum += torch.softmax(model(imgs.flip(-1)), dim=1)

    # Brightness variations
    for factor in [0.85, 1.15]:
        probs_sum += torch.softmax(model((imgs * factor).clamp(0, 1)), dim=1)

    # Slight crop (center)
    _, _, H, W = imgs.shape
    cr = int(0.1 * min(H, W))
    imgs_crop = imgs[:, :, cr:H-cr, cr:W-cr]
    imgs_resized = torch.nn.functional.interpolate(imgs_crop, size=(H, W), mode='bilinear', align_corners=False)
    probs_sum += torch.softmax(model(imgs_resized), dim=1)

    return probs_sum / 6.0  # Trung bình 6 augmentations


# Đánh giá với TTA
print('🔮 Đánh giá với TTA...')
all_preds_tta, all_labels_tta = [], []

model.load_state_dict(best_weights)
for imgs, labels in test_loader:
    probs = predict_tta(model, imgs, CFG['device'])
    preds = probs.argmax(1).cpu().numpy()
    all_preds_tta.extend(preds)
    all_labels_tta.extend(labels.numpy())

from sklearn.metrics import accuracy_score
acc_no_tta = accuracy_score(all_labels, all_preds)
acc_tta    = accuracy_score(all_labels_tta, all_preds_tta)

print(f'\n📊 Kết quả:')
print(f'   Accuracy (no TTA): {acc_no_tta*100:.2f}%')
print(f'   Accuracy (TTA)   : {acc_tta*100:.2f}%')
print(f'   TTA gain         : +{(acc_tta - acc_no_tta)*100:.2f}%')

## 📌 Bước 12: Export Model — TorchScript cho Integration

In [ ]:
model.load_state_dict(best_weights)
model.eval().cpu()

export_dir  = Path(CFG['save_dir'])
dummy_input = torch.randn(1, 3, 256, 128)

# ── 1. TorchScript ──────────────────────────────────────────
try:
    scripted  = torch.jit.trace(model, dummy_input)
    ts_path   = export_dir / 'rolenet_v2_scripted.pt'
    scripted.save(str(ts_path))
    sz = ts_path.stat().st_size / 1024 / 1024
    print(f'✅ TorchScript → {ts_path} ({sz:.1f} MB)')
except Exception as e:
    print(f'⚠️  TorchScript failed: {e}')

# ── 2. Metadata ──────────────────────────────────────────────
metadata = {
    'model'        : 'EfficientNet-B2',
    'version'      : 'v2',
    'num_classes'  : CFG['num_classes'],
    'classes'      : CLASS_NAMES,
    'class_to_idx' : CLASS_TO_IDX,
    'input_size'   : [256, 128],
    'best_val_acc' : round(best_acc * 100, 2),
    'v1_acc'       : 59.05,
    'improvement'  : round((best_acc * 100 - 59.05), 2),
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std' : IMAGENET_STD,
    'label_map'    : {str(i): c for i, c in enumerate(CLASS_NAMES)},
    'tta_enabled'  : True,
}
meta_path = export_dir / 'rolenet_v2_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✅ Metadata   → {meta_path}')

print(f'\n🎉 Export hoàn tất!')
print(f'\n📥 Files cần download:')
for f in sorted(export_dir.iterdir()):
    sz = f.stat().st_size / 1024 / 1024
    print(f'   {f.name:35s} {sz:8.1f} MB')

## 📌 Bước 13: Download Files

In [ ]:
from google.colab import files
import shutil

# Copy charts
for f in ['training_v2.png', 'confusion_v2.png']:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, str(save_path / f))
        print(f'✅ Saved {f} → Drive')

print('\n📥 Downloading model files...')
for fname in ['rolenet_v2_best.pt', 'rolenet_v2_scripted.pt', 'rolenet_v2_metadata.json']:
    fpath = export_dir / fname
    if fpath.exists():
        print(f'   Downloading {fname}...')
        files.download(str(fpath))

print('\n✅ Xong! Lưu các file vào thư mục models/ trong project.')
print('   Sau đó update role_classifier.py để dùng rolenet_v2_best.pt')

---
## 📌 Bước 14: Hướng Dẫn Tích Hợp Về Máy

Sau khi download xong, chạy lệnh sau để update role_classifier:
```bash
# Copy model mới vào thư mục models/
copy rolenet_v2_best.pt      c:\code\camera-ai\models\
copy rolenet_v2_metadata.json c:\code\camera-ai\models\
copy rolenet_v2_scripted.pt  c:\code\camera-ai\models\

# Chạy test tích hợp
python tests/test_rolenet.py
python tests/test_pipeline_smoke.py
```

Sau đó update `MODEL_PT_NAME` trong `modules/role_classifier.py`:
```python
MODEL_PT_NAME   = 'rolenet_v2_best.pt'
MODEL_META_NAME = 'rolenet_v2_metadata.json'
```